# Predicting anomalous events on Taxi dataset

***Unsupervised anomaly detection on timeseries data a Random Cut Forest algorithm.***

Amazon SageMaker Random Cut Forest (RCF) is an algorithm designed to detect anomalous data points within a dataset. Examples of when anomalies are important to detect include when website activity uncharactersitically spikes, when temperature data diverges from a periodic behavior, or when changes to public transit ridership reflect the occurrence of a special event. 

There are some well know anomalies like NYC marathon, Thanksgiving, Christmas, New Year's day, and on the day of a snow storm. We will then use this model to predict anomalous events by emitting an "anomaly score" for each data point. 

# 1. Data Exploratory

In [ ]:
import boto3
import botocore
import sagemaker
import sys

import matplotlib
import matplotlib.pyplot as plt

import pandas as pd
import urllib.request

In [ ]:
execution_role = sagemaker.get_execution_role()

bucket = 'alexbc.datalake'
prefix = 'machine-learning-models/sagemaker/rcf-benchmarks'

data_filename = 'passengers_by_halfhour.csv'
data_source = 'https://s3.amazonaws.com/alexbc.datalake/ny-taxi-data/taxi_datamart/' + data_filename
s3_train_data = 's3://alexbc.datalake/machine-learning-models/sagemaker/rcf-benchmarks/train/data.pbr'

boto3.Session().client('s3').head_bucket(Bucket=bucket)

urllib.request.urlretrieve(data_source, data_filename)
taxi_data = pd.read_csv(data_filename, delimiter=',')
    
print('Training input/output will be stored in: s3://{}/{}'.format(bucket, prefix))

Before training any models it is important to inspect our data, first.The raw data looks like this:

In [ ]:
taxi_data.head()

In [ ]:
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 100

taxi_data.plot(x='timestamp',y='value', title="Volume of Taxi Ridesin 6 months");

In [ ]:
taxi_data[5500:6500].plot(title="Data from 20 days");

In [ ]:
#taxi_data[5952:6003]

# 2. Training

In [ ]:
session = sagemaker.Session()

container = "382416733822.dkr.ecr.us-east-1.amazonaws.com/randomcutforest:latest"

# specify general training job information
rcf = sagemaker.estimator.Estimator(
    container,
    execution_role,
    output_path='s3://{}/{}/output'.format(bucket, prefix),
    train_instance_count=1,
    train_instance_type='ml.m4.xlarge',
    sagemaker_session=session,
)

# set algorithm-specific hyperparameters
rcf.set_hyperparameters(
    num_samples_per_tree = 200,
    num_trees = 50,
    feature_dim = 1,
)

# RCF training requires sharded data. See documentation for
# more information.

s3_train_input = sagemaker.session.s3_input(
    s3_train_data,
    distribution='ShardedByS3Key',
    content_type='application/x-recordio-protobuf',
)

# run the training job on input data stored in S3
rcf.fit({'train': s3_train_input})

In [ ]:
print('Training job name: {}'.format(rcf.latest_training_job.job_name))

# 3. Model Deployment

In [ ]:
rcf_inference = rcf.deploy(
    initial_instance_count=1,
    instance_type='ml.m4.xlarge',
)

print('Endpoint name: {}'.format(rcf_inference.endpoint))

# 4. Inference - Computing Anomaly Scores

In [ ]:
from sagemaker.predictor import csv_serializer, json_deserializer

rcf_inference.content_type = 'text/csv'
rcf_inference.serializer = csv_serializer
rcf_inference.deserializer = json_deserializer

taxi_data_numpy = taxi_data.value.as_matrix().reshape(-1,1)

results = rcf_inference.predict(taxi_data_numpy)
scores = [datum['score'] for datum in results['scores']]

# add scores to taxi data frame and print first few values
taxi_data['score'] = pd.Series(scores, index=taxi_data.index)
taxi_data.head()

The following is a list of known anomalous events which occurred in New York City within this timeframe:

* `2014-11-02` - NYC Marathon
* `2015-01-01` - New Year's Eve
* `2015-01-27` - Snowstorm

Note that our algorithm managed to capture these events along with quite a few others. Below we add these anomalies to the score plot.

In [ ]:
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()

start, end = 0, len(taxi_data)
#start, end = 5500, 6500
taxi_data_subset = taxi_data[start:end]

ax1.plot(taxi_data_subset['value'], color='C0', alpha=0.8)
ax2.plot(taxi_data_subset['score'], color='C1')

ax1.grid(which='major', axis='both')

ax1.set_ylabel('Taxi Ridership', color='C0')
ax2.set_ylabel('Anomaly Score', color='C1')

ax1.tick_params('y', colors='C0')
ax2.tick_params('y', colors='C1')

ax1.set_ylim(0, 40000)
ax2.set_ylim(min(scores), 1.4*max(scores))
fig.set_figwidth(10)

score_mean = taxi_data['score'].mean()
score_std = taxi_data['score'].std()
score_cutoff = score_mean + 3*score_std

anomalies = taxi_data_subset[taxi_data_subset['score'] > score_cutoff]
anomalies

ax2.plot(anomalies.index, anomalies.score, 'ko')

## 4. Stop and Delete the Endpoint

In [ ]:
sagemaker.Session().delete_endpoint(rcf_inference.endpoint)